# Session 10 · Homework Solutions (Teacher Copy) — Logistic vs KNN

**Machine Learning Foundations · Sanketana School of Code**

Worked solution with commentary. On this split logistic regression scores about **0.92** and KNN (k=15) about **0.87** — logistic wins *here*. The teaching point is **comparing with numbers**, not that logistic always wins; a different split can narrow or flip the gap, which is itself worth saying to students.

**Acceptable variation:** either model may look better on a different `random_state`; any correctly-read near-0.5 student passes; the two written sentences must cite the numbers.

## Step 1 · Load, scale, split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

students = pd.read_csv("../../../datasets/anchor/student_habits.csv")
habits = ["study_hours_per_week", "attendance_pct", "sleep_hours_per_night",
          "screen_time_hours_per_day", "practice_sessions_per_week"]
X = students[habits].values
y = students["passed"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

## Step 2 · Fit both models on the same split

Because both models train and test on identical rows, the accuracy comparison is fair.

In [ ]:
logit = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
print("logistic regression test accuracy:", round(logit.score(X_test_s, y_test), 3))   # ~0.92

knn = KNeighborsClassifier(n_neighbors=15).fit(X_train_s, y_train)
print("KNN (k=15) test accuracy:        ", round(knn.score(X_test_s, y_test), 3))       # ~0.87

## Step 3 · Read the probabilities

The probability is logistic regression's real gift: a per-student confidence, not just a vote.

In [ ]:
p_pass = logit.predict_proba(X_test_s)[:, 1]
i_conf_pass = int(np.argmax(p_pass))
i_conf_fail = int(np.argmin(p_pass))
i_unsure    = int(np.argmin(np.abs(p_pass - 0.5)))
print("confident PASS  P(pass) =", round(float(p_pass[i_conf_pass]), 3))   # ~0.99
print("confident FAIL  P(pass) =", round(float(p_pass[i_conf_fail]), 3))   # ~0.01
print("least sure      P(pass) =", round(float(p_pass[i_unsure]), 3))      # ~0.50

## Step 4 · ✅ Model answers

1. **Which scored higher?** *"Logistic regression scored about 0.92 on the test set versus KNN's 0.87 on the same split — logistic wins here by roughly 5 percentage points."* — a bare "logistic" with no numbers does **not** pass. (Fair-minded students may add that a different split could narrow the gap — good instinct.)

2. **What does the probability add?** *"KNN only said pass or fail; logistic regression says *how likely* to pass — so we can spot the students it's genuinely unsure about (near 0.5) and treat them differently from the ones it's certain about."*

**Review talking point for Session 11:** we produce a probability and then cut it at **0.5** to decide. But 0.5 is just a default. For that least-sure student near 0.5, a tiny nudge of the cut flips the prediction — and if one kind of mistake costs more than the other, we'd *want* to move it. Sliding the **threshold** and watching predictions flip is next session.